# Stage 8 & 9: Dual-Brain RAG Testing

In [ ]:
!pip install -q groq sentence-transformers torch

In [ ]:
import torch
import json
import os
from sentence_transformers import SentenceTransformer, util
from groq import Groq
from kaggle_secrets import UserSecretsClient

MODEL_NAME = 'BAAI/bge-large-en-v1.5'
MODELS_DIR = '/kaggle/input/dankgpt-models'

if not os.path.exists(os.path.join(MODELS_DIR, 'embeddings_official.pt')):
    print("Searching for databases in /kaggle/input...")
    import glob
    files = glob.glob('/kaggle/input/**/embeddings_official.pt', recursive=True)
    if files:
        MODELS_DIR = os.path.dirname(files[0])

print(f"Loading {MODEL_NAME}...")
model = SentenceTransformer(MODEL_NAME)

print(f"Loading databases from {MODELS_DIR}...")
db_official_emb = torch.load(os.path.join(MODELS_DIR, 'embeddings_official.pt'))
with open(os.path.join(MODELS_DIR, 'metadata_official.json'), 'r', encoding='utf-8') as f:
    db_official_meta = json.load(f)
    
db_community_emb = torch.load(os.path.join(MODELS_DIR, 'embeddings_community.pt'))
with open(os.path.join(MODELS_DIR, 'metadata_community.json'), 'r', encoding='utf-8') as f:
    db_community_meta = json.load(f)

print("Databases loaded successfully!")

In [ ]:
user_secrets = UserSecretsClient()
groq_api_key = user_secrets.get_secret("GROQ_KEY")
client = Groq(api_key=groq_api_key)

In [ ]:
user_question = "how do i move my pets between rooms ?"

print(f"User Question: {user_question}\n")
query_embedding = model.encode(user_question, convert_to_tensor=True)


off_scores = util.cos_sim(query_embedding, db_official_emb)[0]
off_top = torch.topk(off_scores, k=5)
community_guide_context = ""
for idx in off_top[1]:
    raw = db_official_meta[idx].get('raw_data', db_official_meta[idx].get('knowledge', ''))
    community_guide_context += (json.dumps(raw, indent=2) if isinstance(raw, dict) else str(raw)) + "\n\n"


com_scores = util.cos_sim(query_embedding, db_community_emb)[0]
com_top = torch.topk(com_scores, k=5)
community_context = ""
for idx in com_top[1]:
    raw = db_community_meta[idx].get('raw_data', db_community_meta[idx].get('knowledge', ''))
    community_context += (json.dumps(raw, indent=2) if isinstance(raw, dict) else str(raw)) + "\n\n"

prompt = f"""You are DankGPT, an expert AI assistant for the Dank Memer Discord Bot.
Your goal is to answer the user's question accurately, concisely, and with a friendly tone.

You will be provided with two sources of context to answer the question:
1. [COMMUNITY GUIDES]: This is data extracted from community made guides. This information is 100% accurate. You MUST prioritize this information above all else.
2. [COMMUNITY RUMORS]: These are messages extracted from the community Discord server. This information might be outdated, subjective, or factually incorrect. ONLY use this information if the COMMUNITY GUIDES do not fully answer the question. If you use Community info, you MUST warn the user that the information is based on community speculation.

CRITICAL INSTRUCTIONS:
- NEVER hallucinate or make up information. If the answer cannot be deduced from the provided context, explicitly state \"I don't have enough information to answer that.\"
- FORMATTING: Use Discord-flavored Markdown (bolding, italics, bullet points) to make your response easy to read.
- Be concise and directly address the user's question.
- PROVIDE EXACT DETAILS: Never give vague or obvious answers (e.g., "by doing giveaways"). You MUST extract and provide specific quantities, exact amounts, drop rates, and exact command syntaxes (e.g., "by doing X giveaways Y times") if they are present in the Context. Be highly analytical.


[COMMUNITY GUIDES]
{community_guide_context}

[COMMUNITY RUMORS]
{community_context}

[USER QUESTION]
{user_question}
"""

print("-" * 60)

# 5. Generate the Response
completion = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[{"role": "user", "content": prompt}],
    temperature=0.2,
)

print(completion.choices[0].message.content)